# ImprovedGS + C2F + FreGS-lite cho public scene HCM0204

Notebook này chạy trọn pipeline VAI: preprocess, train ImprovedGS, render test poses, đánh giá ground truth và đóng gói kết quả.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

PHASE_DIR = Path('/kaggle/input/datasets/xuanph/phase1/phase1')
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'Improved-GS'
REPO_BRANCH = 'agent/fregs-lite'

In [ ]:
# ========== CHINH TOAN BO THAM SO CHAY TAI CELL NAY ==========
SET_NAME = 'public_set'
SCENE_NAME = 'HCM0204'
PUBLIC_SET = PHASE_DIR / SET_NAME
RUNTIME_CONFIG_PATH = WORK_ROOT / 'vai_hcm0204.runtime.json'

VAI_CONFIG = {
    'data_root': str(WORK_ROOT / 'vai_cleaned'),
    'output_root': str(WORK_ROOT / 'vai_models'),
    'repeat': 1,
    'python_executable': sys.executable,
    'gpu_auto_select': False,
    'gpu_id': 0,
    'stop_on_error': True,
    'run_postprocess': True,
    'postprocess_script': 'vai_render.py',
    'select_best_repeat_by_psnr': False,
    'train_args': {
        'training_method': 'improvedgs',
        'iterations': 30000,
        'coarse_to_fine': True,
        'coarse_to_fine_middle_iter': 2000,
        'coarse_to_fine_full_iter': 5000,
        'fregs_lite': True,
        'fregs_weight': 0.01,
        'fregs_phase_weight': 0.1,
        'fregs_detail_weight': 1.0,
        'fregs_start_iter': -1,
        'fregs_until_iter': -1,
        'fregs_interval': 1,
        'fregs_low_radius': 0.15,
        'fregs_middle_radius': 0.5,
        'fregs_hann_window': True,
        'eval': False,
        'resolution': -1,
        'data_device': 'cpu',
        'budget': 3000000,
        'progress_bar_width': 100,
        'report_lpips_test': False,
        'report_lpips_train': False,
    },
    'postprocess_args': {
        'output_root': str(WORK_ROOT / 'vai_renders'),
        'eval_root': str(WORK_ROOT / 'vai_eval'),
        'output_extension': 'csv',
        'redistort_interpolation': 'bicubic',
        'sharpen_amount': 1.0,
        'sharpen_sigma': 0.60,
        'jpeg_quality': 95,
        'jpeg_subsampling': 2,
        'evaluate': True,
        'require_gt': True,
        'overwrite': True,
        'psnr_max': 40.0,
        'lpips_net': 'alex',
    },
    'scenes': [{'name': SCENE_NAME}],
}

assert (PUBLIC_SET / SCENE_NAME).is_dir(), f'Khong tim thay {SCENE_NAME}: {PUBLIC_SET}'
RUNTIME_CONFIG_PATH.write_text(
    json.dumps(VAI_CONFIG, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'Runtime config: {RUNTIME_CONFIG_PATH}')
print(json.dumps(VAI_CONFIG, ensure_ascii=False, indent=2))

In [ ]:
# Clone code chinh cua su phu.
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--recursive', '--branch', REPO_BRANCH,
        'https://github.com/mdd206/Improved-GS.git', str(REPO_DIR)
    ], check=True)
os.chdir(REPO_DIR)
print(Path.cwd())

In [ ]:
# Cai COLMAP neu Kaggle image chua co san.
if shutil.which('colmap') is None:
    apt_env = os.environ.copy()
    apt_env['DEBIAN_FRONTEND'] = 'noninteractive'
    subprocess.run(['apt-get', 'update', '-qq'], check=True, env=apt_env)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'colmap'],
                   check=True, env=apt_env)
if shutil.which('colmap') is None:
    raise RuntimeError('Khong the cai COLMAP CLI tren Kaggle')
print('COLMAP:', shutil.which('colmap'))

# Cai dependency Python va CUDA extension.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy==1.26.1', 'opencv-python==4.10.0.82',
                'setuptools==69.5.1', 'tqdm', 'plyfile'], check=True)
for package_dir in [
    'submodules/diff-gaussian-rasterization',
    'submodules/simple-knn',
    'submodules/fused-ssim',
]:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    package_dir, '--no-build-isolation'], check=True)

In [ ]:
# Chuyen SIMPLE_RADIAL thanh PINHOLE RGBA cho HCM0204.
subprocess.run([
    sys.executable, 'vai_preprocess.py',
    '--input', str(PUBLIC_SET),
    '--output', str(WORK_ROOT / 'vai_cleaned'),
    '--subset', SCENE_NAME,
], check=True)

In [ ]:
# In command de kiem tra path va tham so truoc khi train.
subprocess.run([
    sys.executable, 'run.py',
    '-c', str(RUNTIME_CONFIG_PATH),
    '--dry_run',
], check=True)

In [ ]:
# Train ImprovedGS, render 60 test pose va tinh metric public.
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([
    sys.executable, 'run.py',
    '-c', str(RUNTIME_CONFIG_PATH),
], check=True, env=env)

In [ ]:
# Validate ten, kich thuoc va tao ZIP render.
subprocess.run([
    sys.executable, 'vai_package.py',
    '--phase_dir', str(PHASE_DIR),
    '--set_name', SET_NAME,
    '--submission_dir', VAI_CONFIG['postprocess_args']['output_root'],
    '--zip_path', str(WORK_ROOT / f'{SCENE_NAME}_render.zip'),
    '--subset', SCENE_NAME,
    '--output_extension', VAI_CONFIG['postprocess_args']['output_extension'],
], check=True)

# Dong goi JSON evaluation rieng de tai ve.
shutil.make_archive(
    str(WORK_ROOT / f'{SCENE_NAME}_eval'),
    'zip',
    root_dir=VAI_CONFIG['postprocess_args']['eval_root'],
)
print(WORK_ROOT / f'{SCENE_NAME}_render.zip')
print(WORK_ROOT / f'{SCENE_NAME}_eval.zip')